# DSFB Endoduction: NASA IMS Bearing Structural Residual Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-endoduction/dsfb_endoduction_nasa_bearings_colab.ipynb)

---

**Crate:** `dsfb-endoduction`  
**Paper:** *The Thermodynamic Precursor Visibility Principle: A Formal Connection Between Non-Equilibrium Thermodynamics, Critical Phenomena, and Structural Semiotic Inference*  
**Dataset:** NASA IMS Bearing Run-to-Failure (real sensor data only)

---

## 1. Scientific Objective

This notebook implements and evaluates an empirical test of the **Thermodynamic Precursor Visibility Principle** on real run-to-failure bearing data.

### The Question

> Does DSFB-style structured residual analysis reveal degradation precursors **earlier** or **more clearly** than conventional scalar diagnostics?

### The Principle (Informal)

Physical systems approaching critical transitions exhibit precursor signatures — increased variance, autocorrelation growth, spectral redistribution — arising from non-equilibrium thermodynamic constraints. The principle hypothesises that these signatures are **computationally accessible** to regime-aware structural inference, operating on the organisation of residuals rather than scalar summaries alone.

### Epistemic Posture

- The principle is **falsifiable** and **theoretically motivated**, not a proven theorem.
- This notebook **tests** the principle; it does not prove it.
- Results are reported honestly regardless of whether they support the hypothesis.
- No overclaiming language is used.

## 2. Endoduction Framework

### Residual Construction

Given observed signal $x_{\text{obs}}(t)$ and a nominal model $x_{\text{model}}(t)$ estimated from the early-life regime:

$$r(t) = x_{\text{obs}}(t) - x_{\text{model}}(t)$$

### Admissibility Envelope

The admissibility envelope $\mathcal{E}_R$ defines the set of residual behaviours consistent with the nominal regime $R$:

$$\mathcal{E}_R = \{ r : |r_i| \leq k \cdot \sigma_i \;\forall\; i \}$$

where $\sigma_i$ is the per-sample standard deviation from the nominal window and $k$ is set from the configured quantile level.

### Structural Grammar

Residual dynamics are characterised by structural motifs:

| Motif | Meaning |
|---|---|
| **Drift** | Systematic linear trend in residual |
| **Slew** | Maximum absolute rate of change |
| **Persistence** | Longest sign-consistent run (critical slowing proxy) |
| **Variance growth** | Ratio of local variance to nominal baseline |
| **Autocorrelation growth** | Shift in lag-1 autocorrelation from baseline |
| **Spectral shift** | Change in spectral centroid |
| **Breach density** | Fraction of residual outside the envelope |

### Trust / Precursor Score

A bounded scalar $T \in [0, 1]$ aggregating structural evidence:

$$T = \sum_j w_j \cdot \sigma\bigl(f_j(\text{indicators})\bigr)$$

where $\sigma$ is the logistic sigmoid. This is **not** a probability of failure — it is a departure-from-nominal structure score.

### Lead Time

The empirical lead time $\tau$ is the interval between first sustained DSFB detection and the failure reference window:

$$\tau = t_{\text{failure}} - t_{\text{first\_sustained\_detection}}$$

### What This Does Not Prove

- Does not prove the Thermodynamic Precursor Visibility Principle
- Does not establish universality across all systems
- Does not measure thermodynamic entropy production directly
- Does not replace validated condition monitoring systems

## 3. Environment Setup

Install the Rust toolchain and clone the repository.

In [ ]:
import os, subprocess, sys

# Install Rust toolchain
if not os.path.exists(os.path.expanduser('~/.cargo/bin/rustc')):
    print('Installing Rust toolchain...')
    subprocess.check_call('curl https://sh.rustup.rs -sSf | sh -s -- -y', shell=True)
else:
    print('Rust toolchain already installed.')

os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
subprocess.check_call(['rustc', '--version'])
subprocess.check_call(['cargo', '--version'])

In [ ]:
# Clone the repository
REPO_DIR = '/content/dsfb'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/infinityabundance/dsfb.git', REPO_DIR])
else:
    print('Repository already cloned.')
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 4. Build the Crate

In [ ]:
# Build in release mode
subprocess.check_call(['cargo', 'build', '-p', 'dsfb-endoduction', '--release'])
print('Build successful.')

## 5. Download & Verify the NASA IMS Bearing Dataset

**Provenance:** NASA Prognostics Data Repository, Intelligent Maintenance Systems (IMS) Center, University of Cincinnati.  
**Reference:** J. Lee, H. Qiu, G. Yu, J. Lin, "Rexnord Technical Services, IMS, University of Cincinnati. Bearing Data Set", NASA Prognostics Data Repository, 2007.  
**URL:** https://www.nasa.gov/content/prognostics-center-of-excellence-data-set-repository

The dataset contains run-to-failure vibration measurements from bearings under constant radial load at 2156 RPM, sampled at 20 kHz.

In [ ]:
# The crate handles dataset download via the --download flag.
# The data will be stored in a 'data' directory relative to the repo root.
DATA_ROOT = os.path.join(REPO_DIR, 'crates', 'dsfb-endoduction', 'data')
os.makedirs(DATA_ROOT, exist_ok=True)
print(f'Data root: {DATA_ROOT}')

## 6. Run the Full Analysis Pipeline

This executes the complete DSFB endoduction pipeline:
1. Load and parse bearing data
2. Estimate nominal baseline
3. Compute residuals and admissibility envelope
4. Extract structural motifs
5. Compute trust/precursor score
6. Compare against classical baselines
7. Generate figures, CSV, PDF report, and ZIP bundle

In [ ]:
OUTPUT_DIR = os.path.join(REPO_DIR, 'crates', 'dsfb-endoduction', 'output-dsfb-endoduction')

result = subprocess.run(
    ['cargo', 'run', '-p', 'dsfb-endoduction', '--release', '--',
     'run',
     '--data-root', DATA_ROOT,
     '--bearing-set', '1',
     '--channel', '0',
     '--download',
     '--output-dir', OUTPUT_DIR],
    capture_output=True, text=True
)

print('=== STDOUT ===')
print(result.stdout)
print('=== STDERR ===')
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'Pipeline failed with return code {result.returncode}')

# The last line of stdout is the run directory path
RUN_DIR = result.stdout.strip().split('\n')[-1].strip()
print(f'\nRun output directory: {RUN_DIR}')

## 7. Verify Outputs

Check that all required artifacts were produced.

In [ ]:
import json

# Load and display manifest
manifest_path = os.path.join(RUN_DIR, 'manifest.json')
with open(manifest_path) as f:
    manifest = json.load(f)

print('=== ACCEPTANCE GATES ===')
for gate, status in manifest['gates'].items():
    symbol = '✓' if status else '✗'
    print(f'  {symbol} {gate}: {status}')

print(f"\nFiles produced: {len(manifest['files_produced'])}")
for f in manifest['files_produced']:
    print(f'  - {f}')

print(f"\nSnapshots processed: {manifest['snapshots_processed']}")
print(f"Nominal windows: {manifest['nominal_windows']}")
print(f"DSFB first detection: {manifest['dsfb_first_detection']}")

## 8. Summary Metrics

In [ ]:
summary = manifest['summary']
print('=== SUMMARY METRICS ===')
print(f"Total windows:         {summary['total_windows']}")
print(f"Nominal end window:    {summary['nominal_end_window']}")
print(f"Failure window:        {summary['failure_window']}")
print(f"DSFB lead time:        {summary['dsfb_lead_time_windows']} windows")
print()
print('=== BASELINE LEAD TIMES ===')
for method, lead in sorted(summary['baseline_lead_times'].items()):
    det = summary['baseline_first_detections'].get(method)
    print(f"  {method:25s}  detection: {str(det):>8s}  lead: {str(lead):>8s} windows")

## 9. Figures

All 12 publication-grade figures, displayed inline.

In [ ]:
from IPython.display import display, Image

figure_files = sorted([f for f in os.listdir(RUN_DIR) if f.endswith('.png')])
print(f'Displaying {len(figure_files)} figures:\n')

for fig in figure_files:
    fig_path = os.path.join(RUN_DIR, fig)
    print(f'--- {fig} ---')
    display(Image(filename=fig_path))
    print()

## 10. Metrics Table

In [ ]:
import csv as csv_mod

csv_path = os.path.join(RUN_DIR, 'metrics.csv')
with open(csv_path) as f:
    reader = csv_mod.DictReader(f)
    rows = list(reader)

# Show first and last few rows
print(f'Total rows: {len(rows)}')
print(f'Columns: {list(rows[0].keys())}')
print()

# Print summary table: first 5 and last 5 rows, key columns
key_cols = ['index', 'rms', 'kurtosis', 'envelope_breach_fraction', 'trust_score']
header = '  '.join(f'{c:>25s}' for c in key_cols)
print(header)
print('-' * len(header))
for row in rows[:5] + rows[-5:]:
    vals = '  '.join(f'{row[c]:>25s}' for c in key_cols)
    print(vals)

## 11. Download Artifacts

Download the PDF report and ZIP bundle containing all outputs.

In [ ]:
try:
    from google.colab import files as colab_files
    COLAB = True
except ImportError:
    COLAB = False

pdf_path = os.path.join(RUN_DIR, 'report.pdf')
zip_path = os.path.join(RUN_DIR, 'bundle.zip')

if COLAB:
    print('Offering PDF for download...')
    colab_files.download(pdf_path)
    print('Offering ZIP for download...')
    colab_files.download(zip_path)
else:
    from IPython.display import display, HTML
    import base64

    def download_button(filepath, label):
        with open(filepath, 'rb') as f:
            data = base64.b64encode(f.read()).decode()
        fname = os.path.basename(filepath)
        html = f'<a download="{fname}" href="data:application/octet-stream;base64,{data}" style="padding:8px 16px;background:#1a73e8;color:white;border-radius:4px;text-decoration:none;font-weight:bold;">{label}</a>'
        display(HTML(html))

    download_button(pdf_path, 'Download PDF Report')
    print()
    download_button(zip_path, 'Download ZIP Bundle')

---

## Interpretation Notes

- Results above test whether DSFB structural residual analysis provides earlier precursor detection than classical scalar diagnostics on the NASA IMS dataset.
- The trust score is a departure-from-nominal proxy, not a direct thermodynamic measurement.
- Lead-time estimates are sensitive to threshold and sustained-count parameters; see the robustness figure.
- If DSFB does not outperform a baseline on some metric, that is reported faithfully.
- This evaluation supports but does not prove the Thermodynamic Precursor Visibility Principle.

---

*Generated by `dsfb-endoduction` — [github.com/infinityabundance/dsfb](https://github.com/infinityabundance/dsfb)*